In [ ]:
# ========== 第 4 天 Pythonista：FlightAI + Tool Calling + SQLite + Gradio ==========
# 练习目标：让模型通过 tools 读写本地票价数据库，并在 Gradio 聊天界面里多轮调用

# ========== 区块 1 — 导入依赖 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：把 tool_calls 里的 arguments（JSON 字符串）解析成 dict
import json
# 导入标准库 sqlite3：用本地文件做轻量数据库（SQLite）
import sqlite3
# 从 dotenv 导入 load_dotenv：从 .env 加载密钥，避免写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：官方云端；也可配合 base_url 指向 Ollama 兼容接口
from openai import OpenAI
# 导入 gradio：在本地快速搭一个聊天 UI
import gradio as gr

In [ ]:
# ========== 区块 2 — 初始化：API Key、模型名、客户端 ==========

# override=True：.env 覆盖进程中已有同名环境变量
load_dotenv(override=True)

# 从环境读取 OPENAI_API_KEY（不要把完整密钥打印出来）
openai_api_key = os.getenv("OPENAI_API_KEY")
# 有密钥：只展示前 8 位做存在性检查
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    # 无密钥：提示未设置（打印文案保持原样）
    print("OpenAI API Key not set")

# 选用的云端模型 id（字符串保持原样）
MODEL = "gpt-4.1-mini"
# 创建 OpenAI 客户端：默认从环境变量取 key
openai = OpenAI()

# --- 备选：改用本地 Ollama（OpenAI 兼容模式）时取消下面两行注释，并注释掉上面的 MODEL/openai ---
# MODEL = "llama3.1:8b"
# openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
# ========== 区块 3 — system_message：航空助手人设与回答风格 ==========
# 发给模型的 system 文本保持英文原样（影响回答行为，不翻译）

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
# ========== 区块 4 — SQLite：数据库文件与 prices 表 ==========

# DB：SQLite 文件名；不存在时会在首次连接时创建
DB = "prices.db"

# with 连接：退出块时自动关闭连接
with sqlite3.connect(DB) as conn:
    # cursor：用来执行 SQL 语句
    cursor = conn.cursor()
    # 若表不存在则创建：city 主键，price 为实数
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    # commit：把建表结果持久化到文件
    conn.commit()

In [ ]:
# ========== 区块 5 — 真正的 Tools：Python 函数读写票价 ==========
# 这些函数会被模型通过 tool_calls「点名」执行；返回值是给模型看的字符串

def get_ticket_price(city: str) -> str:
    """Tool：按城市查 prices 表票价；返回给 LLM 使用的说明字符串。"""
    # flush=True：日志立刻刷到终端，方便调试「工具是否被调用」
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)

    # 打开 SQLite 连接并查询
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # 参数化查询（? 占位）避免 SQL 注入；city 统一小写便于匹配
        cursor.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.lower(),)  # 单元素 tuple，末尾逗号必需
        )
        # fetchone：得到 (price,) 或 None
        result = cursor.fetchone()

    # 查到行：拼一句英文票价说明（字符串给模型，保持原样）
    if result:
        return f"Ticket price to {city} is ${result[0]}"
    # 未查到：返回「无数据」说明（保持原样）
    return "No price data available for this city"


def set_ticket_price(city: str, price: float) -> str:
    """Tool：插入或更新某城市票价；返回确认字符串。"""
    # 调试日志：确认 set 工具被调用及参数
    print(f"DATABASE TOOL CALLED: Setting price for {city} -> {price}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # UPSERT：有则更新 price，无则插入；三个 ? 对应下面三个绑定值
        cursor.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) "
            "ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price)  # insert 的 city/price + update 的 price
        )
        # 提交写入，保证落盘
        conn.commit()

    # 返回确认句，供下一轮对话里模型引用
    return f"Updated ticket price for {city} to ${price}"

In [ ]:
# ========== 区块 6 —（可选）种子数据：预填几座城市的票价 ==========

# seed_prices：示例字典 city -> 数值价格
seed_prices = {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "madrid":1500,
    "Hondarribia":1970
}

# 遍历字典，复用 set_ticket_price 写入/更新数据库
for city, price in seed_prices.items():
    set_ticket_price(city, price)

In [ ]:
# ========== 区块 7 — Tool schemas：把函数能力「说明书」交给模型 ==========
# 重要：schema 里参数名必须与 Python 函数形参一致，后面才能 func(**args)

# get_ticket_price(city) 的 JSON schema
get_price_schema = {
    "name": "get_ticket_price",
    # description 会进模型上下文，保持英文原样
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            # 键名必须是 city，与函数参数一致
            "city": {
                "type": "string",
                "description": "Destination city (e.g. London, Paris)"
            }
        },
        "required": ["city"],  # city 必填
        "additionalProperties": False  # 禁止额外未知参数
    }
}

# set_ticket_price(city, price) 的 JSON schema
set_price_schema = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Destination city (e.g. London, Paris)"
            },
            # 键名必须是 price，与函数参数一致
            "price": {
                "type": "number",
                "description": "Ticket price (numeric), e.g. 899"
            }
        },
        "required": ["city", "price"],  # 两参数都必填
        "additionalProperties": False
    }
}

# tools：传给 chat.completions.create(..., tools=tools) 的列表
tools = [
    {"type": "function", "function": get_price_schema},
    {"type": "function", "function": set_price_schema},
]

In [ ]:
# ========== 区块 8 — TOOL_REGISTRY：工具名 → 可执行函数（白名单） ==========
# 用字典分发，避免对每个 tool 写一长串 if/elif；未登记的名字一律不执行

TOOL_REGISTRY = {
    # 授权查询票价
    "get_ticket_price": get_ticket_price,
    # 授权设置/更新票价
    "set_ticket_price": set_ticket_price,
}

In [ ]:
# ========== 区块 9 — handle_tool_calls：通用工具调用处理器 ==========

def handle_tool_calls(assistant_message):
    """
    接收带 tool_calls 的助手消息：按白名单执行每个调用，
    返回若干 role="tool" 的消息，以便追加进对话历史。
    """
    # 累积每条 tool 的回复消息
    tool_responses = []

    # 遍历模型请求的每一个 tool_call
    for tool_call in assistant_message.tool_calls:
        # 模型点名的函数名（字符串）
        tool_name = tool_call.function.name
        # 在白名单里查找对应 Python 函数；没有则为 None
        func = TOOL_REGISTRY.get(tool_name)

        # arguments 是 JSON 字符串；空/None 时当成 {}
        raw_args = tool_call.function.arguments or "{}"
        # JSON -> dict，供 **kwargs 解包
        args = json.loads(raw_args)

        if func is None:
            # 未注册工具：返回安全提示，不执行任意代码
            output = f"Tool '{tool_name}' is not available."
        else:
            # 按 schema/实参调用：func(**args)
            output = func(**args)

        # 组装 role=tool 消息；tool_call_id 把结果绑回对应请求
        tool_responses.append({
            "role": "tool",
            "content": output,
            "tool_call_id": tool_call.id
        })

    # 交给 chat() 把这些消息 extend 进 messages
    return tool_responses

In [ ]:
# ========== 区块 10 — chat：Gradio 回调 + 工具调用循环 ==========

def chat(message, history):
    """
    Gradio 每轮用户发言时调用：
    - message：本轮用户输入
    - history：Gradio 提供的历史（user/assistant）
    """
    # 只保留 role/content，去掉 Gradio 可能附带的其它字段
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    # 拼完整 messages：system + 历史 + 本轮 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    # 首次调用模型，并声明可用 tools
    response = openai.chat.completions.create(
        model=MODEL,  # 上面选定的模型
        messages=messages,  # 完整上下文
        tools=tools  # 允许模型发起 tool_calls
    )

    # 若 finish_reason 仍是 tool_calls：执行工具 → 把结果喂回模型，直到给出最终文本
    while response.choices[0].finish_reason == "tool_calls":
        # 含 tool_calls 的助手消息
        assistant_msg = response.choices[0].message
        # 执行工具，得到 role=tool 消息列表
        tool_msgs = handle_tool_calls(assistant_msg)

        # 先追加助手的「要调用工具」消息
        messages.append(assistant_msg)
        # 再追加工具结果
        messages.extend(tool_msgs)

        # 带着工具结果再次请求模型
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    # 不再要工具时：返回最终助手文本给 Gradio 显示
    return response.choices[0].message.content

In [ ]:
# ========== 区块 11 — 启动 Gradio 聊天界面 ==========
# type="messages"：使用 messages 格式的历史（与上面 chat 的 history 结构匹配）

gr.ChatInterface(fn=chat, type="messages").launch()